In [3]:
import pandas as pd
from pathlib import Path


def add_prompt_column(csv_file, output_file=None, prompt_col="Prompt"):
    """
    Read a CSV file and add a prompt column based on:
    - 'Question'
    - 'Group'
    - columns starting with 'Option '

    The letters in 'Group' determine:
    - allowed_letters
    - which options are included in the prompt

    Example:
    Group = "ABCD" means include only options A, B, C, D.
    """

    csv_file = Path(csv_file)
    df = pd.read_csv(csv_file, encoding="utf-8-sig")

    required_cols = ["Question", "Group"]
    for col in required_cols:
        if col not in df.columns:
            raise ValueError(f"Missing required column: {col}")

    option_cols = [col for col in df.columns if col.startswith("Option ")]
    if not option_cols:
        raise ValueError("No columns starting with 'Option ' were found.")

    # Build mapping like:
    # "A" -> "Option A"
    # "B" -> "Option B"
    # ...
    option_col_map = {}

    for col in option_cols:
        suffix = col.replace("Option ", "").strip().upper()

        # Works for columns like "Option A", "Option B", ...
        if suffix in ["A", "B", "C", "D", "E","F"]:
            option_col_map[suffix] = col

    # Fallback: if columns are like Option 1, Option 2, ...
    # map them by order to A, B, C, D, E
    if not option_col_map:
        letters = ["A", "B", "C", "D", "E","F"]
        option_col_map = {
            letter: col
            for letter, col in zip(letters, option_cols)
        }

    def clean_text(value):
        if pd.isna(value):
            return ""
        return str(value).strip()

    def make_prompt(row):
        question = clean_text(row["Question"])

        group = clean_text(row["Group"]).upper()
        allowed_letters = [letter for letter in group if letter in ["A", "B", "C", "D", "E","F"]]

        allowed_letters_text = ", ".join(allowed_letters)

        option_lines = []

        for letter in allowed_letters:
            if letter not in option_col_map:
                raise ValueError(
                    f"Could not find a matching option column for option letter '{letter}'."
                )

            option_text = clean_text(row[option_col_map[letter]])
            option_lines.append(f"{letter}. {option_text}")

        options_text = "\n".join(option_lines)

        prompt = f"""You are an expert medical assistant.
                    Answer the following Arabic medical multiple-choice question.

                    Return only the correct option letter. Return exactly one uppercase letter from the allowed options. No explanation."
                    Allowed options: {allowed_letters_text}

                    Question:
                    {question}

                    Options:
                    {options_text}

                    What is your answer?"""

        return prompt

    df[prompt_col] = df.apply(make_prompt, axis=1)

    if output_file is None:
        output_file = csv_file

    required_cols = [
        "Question Number",
        "Question",
        "Option A",
        "Option B",
        "Option C",
        "Option D",
        "Option E",
        "Option F",
        "Correct Answer",
        "Level",
        "Medical Specialty",
        "Group",
        "Prompt",
    ]
    df = df[required_cols]
    df.to_csv(output_file, index=False, encoding="utf-8-sig")

    return df

In [4]:
path_cleaned_test = "data/cleaned/medarabench_train.csv"
path_test_with_prompts = "data/cleaned/medarabench_train_with_prompts.csv"

add_prompt_column(path_cleaned_test, path_test_with_prompts)

,Question Number,Question,Option A,Option B,Option C,Option D,Option E,Option F,Correct Answer,Level,Medical Specialty,Group,Prompt
0,43,من وظائف القلب كل ما يلي عدا :,يتقلص بشكل ايقاعي,يضخ الدم في جهاز الدوران,يساعد في ضخ اللمف,يفرز العامل الطازج للصوديوم,NaN,NaN,B,Y2,Histology and Anatomy,ABCD,You are an expert medical assistant.\n ...
1,41,كل ما يلى صحيح حول المفاصل عدا :,من المفاصل الكروية المفصل الأخرمي الترقوي.,من المفاصل البكرية تفصل العضد مع الزند.,من المفاصل الثقمية (الاهليلجية) مفصل الرسغ.,من المفاصل المسطحة المفصل الأخرمي الترقوي.,NaN,NaN,A,Y1,Anatomy,ABCD,You are an expert medical assistant.\n ...
2,56,من أقسام العظم الصدغي ما عدا:,الجسم,الخشاء,الصدفة,الصخرة,NaN,NaN,A,Y1,Anatomy,ABCD,You are an expert medical assistant.\n ...
3,161,تقع الثقبة المدورة في العظم :,.Frontal,Sphenoid .,Temporal .,Occipital .,NaN,NaN,B,Y1,Anatomy,ABCD,You are an expert medical assistant.\n ...
4,806,كل ما يلي صحيح عن نوى العصب الوجهي عدا:,نواة تتواسط حاسة الذوق .,نواة لإفراز الدمع.,نواة للتوازن.,نواة حركية.,NaN,NaN,C,Y1,Anatomy,ABCD,You are an expert medical assistant.\n ...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
19886,769,كل مما يلي من العوامل الواقية من سرطان المعدة ...,الخضار.,الفواكه.,فيتامين D.,أسبيرين.,سلينيوم.,NaN,C,Y4,Surgery,ABCDE,You are an expert medical assistant.\n ...
19887,820,تشنج الفؤاد يمتاز بما يلي ماعدا:,اشتداد الحركات الحوية الثانوية.,غياب الحركات الكتلية.,تقطع الحركات الحوية الأساسية.,عدم التناسق في حركات المري الحوية وارتخاء LES,NaN,NaN,A,Y4,Surgery,ABCD,You are an expert medical assistant.\n ...
19888,861,يفيد الإيكوغرافي للثدي في: هنالك عبارة خاطئة و...,تقييم كتل الثدي.,يتفوق الإيكوغرافي على الماموغرافي بخصوص دراسة ...,يستطيع التفريق بين كتلة مصمتة أو كيسيه في الثدي.,يمكن أن يساعد في توجيه الدراسة الخلوية FNA أو ...,يمكن أن يقيم الاستجابة للعلاج الكيماوي والشعاع...,NaN,B,Y4,Surgery,ABCDE,You are an expert medical assistant.\n ...
19889,1079,العبارة الصحيحة حول داء كرون هي:,سرطانة الأمعاء الصغيرة هي من اختلاطاته.,عند استطباب الاستئصال الجراحي يجب أن تكون حواف...,يترافق تصنيع التضيق مع معدل امراضيات أعلى من ا...,يترافق داء كرون حول الشرج مع التهاب الصائم أكث...,يجب أن يُجرى استئصال البواسير أبكر ما يمكن لأن...,NaN,B,Y4,Surgery,ABCDE,You are an expert medical assistant.\n ...
